# Importaciones necesarias

In [1]:
import pandas as pd
import numpy as np


# Carga de datos

Procedemos a cargar los datos de los tres archivos csv.

In [3]:
inventario_df = pd.read_csv('inventarios.csv')

satisfaccion_df = pd.read_csv('satisfaccion.csv')

ventas_df = pd.read_csv('ventas.csv')

Ahora vamos a limpiar los datos eliminando los valores nulos. 

In [4]:
inventario_df = inventario_df.dropna()

satisfaccion_df = satisfaccion_df.dropna()

ventas_df = ventas_df.dropna()

# Exploración de datos

Tras la limpieza realizada vamos a explorar estos datos. Primero vamos a calcular las ventas totales por producto y tienda. 

In [6]:
ventas_por_producto_tienda = ventas_df.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum()
print("Ventas totales (cantidad) por tienda y producto:\n")
display(ventas_por_producto_tienda)

Ventas totales (cantidad) por tienda y producto:



ID_Tienda  Producto  
1          Producto A    20
           Producto B    15
2          Producto A    30
           Producto C    25
3          Producto A    10
           Producto B    40
4          Producto A    25
           Producto C    35
5          Producto B    20
           Producto C    30
Name: Cantidad_Vendida, dtype: int64

Tras esto, pasamos a calcular los ingresos totales por tienda, multiplicando la cantidad vendidad por el precio unitario.

In [7]:
ventas_df['Ingreso_Total'] = ventas_df['Cantidad_Vendida'] * ventas_df['Precio_Unitario']
ingresos_por_tienda = ventas_df.groupby('ID_Tienda')['Ingreso_Total'].sum()
print("\nIngresos totales por tienda:\n")
display(ingresos_por_tienda)


Ingresos totales por tienda:



ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Ingreso_Total, dtype: int64

Seguidos de estos cálculos vamos a ver un resumen estadístico de las ventas.

In [8]:
print("\nResumen estadístico de las ventas:")
display(ventas_df.describe())


Resumen estadístico de las ventas:


,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Ingreso_Total
count,10.000000,10.000000,10.000000,10.000000
mean,3.000000,25.000000,190.000000,5050.000000
std,1.490712,9.128709,87.559504,3361.960407
min,1.000000,10.000000,100.000000,1000.000000
25%,2.000000,20.000000,100.000000,2625.000000
50%,3.000000,25.000000,200.000000,3500.000000
75%,4.000000,30.000000,275.000000,7875.000000
max,5.000000,40.000000,300.000000,10500.000000


Y para terminar esta sección vamos a calcular el promedio de ventas por tienda y categoría de productos.

In [9]:
promedio_ventas_tienda = ventas_df.groupby('ID_Tienda')['Cantidad_Vendida'].mean()
print("\nPromedio de unidades vendidas por transacción, por tienda:\n")
display(promedio_ventas_tienda)


Promedio de unidades vendidas por transacción, por tienda:



ID_Tienda
1    17.5
2    27.5
3    25.0
4    30.0
5    25.0
Name: Cantidad_Vendida, dtype: float64

# Análisis de inventarios

Procedemos a calcular la rotación de inventarios para cada tienda y lo vamos a almacenar en una nueva columna dentro del DataFrame de inventarios.

In [15]:
ventas_totales_tienda = ventas_df.groupby('ID_Tienda')['Cantidad_Vendida'].sum().reset_index()

stock_total_tienda = inventario_df.groupby('ID_Tienda')['Stock_Disponible'].sum().reset_index()

df_inventario_tienda = pd.merge(stock_total_tienda, ventas_totales_tienda, on='ID_Tienda', how='left')

df_inventario_tienda['Cantidad_Vendida'] = df_inventario_tienda['Cantidad_Vendida'].fillna(0)

df_inventario_tienda['Rotacion'] = df_inventario_tienda['Cantidad_Vendida'] / df_inventario_tienda['Stock_Disponible']

print("--- DataFrame con Rotación de Inventarios Global por Tienda ---")
display(df_inventario_tienda)

--- DataFrame con Rotación de Inventarios Global por Tienda ---


,ID_Tienda,Stock_Disponible,Cantidad_Vendida,Rotacion
0,1,90,35,0.388889
1,2,105,55,0.523810
2,3,110,50,0.454545
3,4,120,60,0.500000
4,5,100,50,0.500000


Y ahora pasamos a mostrar aquellas tiendas con niveles críticos de inventario. Es decir, aquellas tiendas donde el porcentaje de productos vendidos sea menor al 10% del stock disponible. 

In [16]:
inventarios_criticos = df_inventario_tienda[df_inventario_tienda['Rotacion'] < 0.1]

print("\n--- Tiendas con niveles críticos de inventario global (Rotación < 10%) ---")
display(inventarios_criticos[['ID_Tienda', 'Rotacion']])


--- Tiendas con niveles críticos de inventario global (Rotación < 10%) ---


,ID_Tienda,Rotacion


# Satisfacción del cliente
Ahora pasamos a la satisfacción del cliente. Para ello vamos a realizar un análisis de estos en cada tienda. 

In [13]:
df_rendimiento = pd.merge(satisfaccion_df, ingresos_por_tienda, on='ID_Tienda', how='left')

# Verificamos si hay tiendas sin ingresos y los reemplazamos por 0
df_rendimiento['Ingreso_Total'] = df_rendimiento['Ingreso_Total'].fillna(0)

print("Tabla de rendimiento unificada (Satisfacción e Ingresos):")
display(df_rendimiento.head())

Tabla de rendimiento unificada (Satisfacción e Ingresos):


,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación,Ingreso_Total
0,1,85,2023-01-15,5000
1,2,90,2023-01-15,10500
2,3,70,2023-01-15,9000
3,4,65,2023-01-15,13000
4,5,55,2023-01-15,13000


Ahora pasamos a analizar cuales son las tiendas con bajos niveles de satisfacción. 

In [14]:
tiendas_criticas = df_rendimiento[df_rendimiento['Satisfacción_Promedio'] < 60]

print("Tiendas con niveles bajos de satisfacción (< 60%):")
display(tiendas_criticas)

Tiendas con niveles bajos de satisfacción (< 60%):


,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación,Ingreso_Total
4,5,55,2023-01-15,13000


### Recomendaciones para mejorar el rendimiento (Tienda 5)

A partir del análisis anterior, observamos que la **Tienda 5** presenta un nivel de satisfacción del 55%, a pesar de generar uno de los ingresos totales más altos. Esto indica que el alto volumen de ventas no se está traduciendo en una buena experiencia para el cliente.

**Recomendaciones propuestas:**
1. **Evaluar la saturación del personal:** El alto volumen de ingresos sugiere una gran afluencia de clientes. Es vital investigar si el personal actual es suficiente o si se están generando largos tiempos de espera en caja y atención, lo que frustra al cliente.
2. **Revisar el flujo y rotura de stock:** Una alta demanda puede provocar que los productos más buscados se agoten rápidamente, generando insatisfacción en los clientes que acuden a la tienda y no encuentran lo que buscan.
3. **Auditoría de calidad y devoluciones:** Comprobar si los productos vendidos en esta sucursal presentan un índice de quejas o devoluciones mayor al resto de tiendas.
4. **Desplegar encuestas en el punto de venta:** Implementar un sistema de *feedback* rápido en la propia tienda para identificar exactamente qué aspecto falla (limpieza, orden, trato del personal o disponibilidad de inventario).

# Operaciones con Numpy
Procedemos a calcular la mediana y desviación estandar de las ventas totales. 

In [17]:
ventas_array = ventas_df['Ingreso_Total'].to_numpy()

# Calcular la mediana y la desviación estándar
mediana_ventas = np.median(ventas_array)
desviacion_ventas = np.std(ventas_array)

print(f"Mediana de las ventas totales: {mediana_ventas}")
print(f"Desviación estándar de las ventas: {desviacion_ventas:.2f}")

Mediana de las ventas totales: 3500.0
Desviación estándar de las ventas: 3189.44


Por último vamos a simular proyecciones de ventas futuras a través de generar arrays aleatorios. 

In [18]:
np.random.seed(42) 
factor_crecimiento = np.random.uniform(0.95, 1.15, size=5)
ventas_proyectadas = ventas_array.mean() * factor_crecimiento

print("\nProyección de ventas promedio para los próximos 5 periodos:")
print(ventas_proyectadas)


Proyección de ventas promedio para los próximos 5 periodos:
[5175.78552004 5757.72144947 5536.81388123 5402.14506904 4955.07882685]
